# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing dataset elements by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '[unknown name]')}: {getattr(metadata, 'description', '[no description]')}")

## 2. Data Overview
Review available record sets and their fields and field IDs. All entities (record set, field, columns) are referenced using their `@id` field for consistency.

Let's list all available record sets and, for each, their fields and columns.

In [ ]:
# List all record sets and their fields/columns by @id
print("Available Record Sets (by @id):")
record_sets = dataset.record_sets  # List of RecordSet objects
record_set_ids = []

for rs in record_sets:
    print(f"- RecordSet @id: {getattr(rs, '@id', getattr(rs, 'id', '[no id]'))} | name: {getattr(rs, 'name', '[no name]')}")
    record_set_ids.append(getattr(rs, '@id', getattr(rs, 'id', '[no id]')))
    print("  Fields (@id):")
    for field in getattr(rs, 'fields', []):
        print(f"    - {getattr(field, '@id', getattr(field, 'id', '[no id]'))}: {getattr(field, 'name', '[no name]')}, type: {getattr(field, 'data_type', '[unknown]')}")
        # Also print columns (for tabular data)
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      - column @id: {getattr(col, '@id', getattr(col, 'id', '[no id]'))}, column name: {getattr(col, 'name', '[no name]')}")
    print("")
if not record_set_ids:
    print("No record sets found.")

## 3. Data Extraction
Load data from each record set into a DataFrame using the record set `@id` from the previous step. All further selection will use these IDs directly.

In [ ]:
# We extract all record sets by @id.
# Feel free to adjust the list to desired record sets
dataframes = {}

for record_set_id in record_set_ids:
    # Use the record_set_id as key throughout
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records into DataFrame for RecordSet @id: '{record_set_id}'")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records loaded for RecordSet @id: '{record_set_id}'")

# Select the first loaded record set for further exploration
main_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes:
        main_record_set_id = rs_id
        break
if main_record_set_id:
    print(f"Proceeding with RecordSet @id: {main_record_set_id}")
else:
    raise RuntimeError("No suitable record set with data found!")

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic data processing using `@id` for fields/columns:

1. Filter records by a numeric column.
2. Normalize a selected numeric column.
3. (If available) Group by a categorical field.

First, we'll automatically select an integer/float column by its `@id`.

In [ ]:
df = dataframes[main_record_set_id]

# Select a numeric field automatically by dtype
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    raise RuntimeError("No numeric field found for analysis!")
print(f"Selected numeric field for EDA (by @id): {numeric_field_id}")

# Filter: threshold for numeric value
threshold = df[numeric_field_id].mean() # default: keep above mean
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records (first five):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to find a categorical column for grouping
group_field_id = None
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() < min(10, len(df)//5):  # Small number of categories
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize distributions and relationships between fields. We use the selected numeric and (if found) group/categorical field for plotting.

All plots include axis labels by `@id` for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True, color='royalblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

# Pairplot if there are at least 2 numeric fields
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if len(numeric_cols) >= 2:
    sns.pairplot(df[numeric_cols].dropna())
    plt.suptitle('Pairplot of Numeric Fields (@id)', y=1.02)
    plt.show()

## 6. Conclusion
In this notebook we:
- Loaded and inspected the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer dataset using its Croissant schema via `mlcroissant`
- Explored available record sets, fields, and columns by their `@id` identifiers
- Loaded data into pandas DataFrames, demonstrated filtering and normalization using field `@id`s
- Provided basic groupwise summaries and visualizations using only `@id` identifiers for transparency

This template can be adapted for any Croissant-style dataset by referencing record sets and fields by `@id` to ensure full reproducibility and clarity.

> _Tip: use the printed lists of record set and field `@id`s for scripting further, more complex analyses in your own work._